# Attention Operator Validation (PYNQ) v1

This notebook verifies QK^T -> softmax -> (softmax * V) using PL GEMM twice.
All outputs are ASCII to avoid encoding issues.


In [ ]:
print('Attention Operator Validation v1')
import time
import numpy as np
import pynq


In [ ]:
ARRAY_ROW = 12
ARRAY_COL = 16

def ceil_to(x, base):
    return ((x + base - 1) // base) * base

def calc_pad(m, k, n):
    m_pad = m if (m % 2 == 0) else (m + 1)
    k_pad = ceil_to(k, ARRAY_ROW)
    n_pad = ceil_to(n, ARRAY_COL)
    return m_pad, k_pad, n_pad

def to_u8(x):
    return int(x) & 0xFF

def u8_to_i8(arr_u8):
    arr_u8 = np.array(arr_u8, dtype=np.uint8)
    return np.where(arr_u8 < 128, arr_u8, arr_u8 - 256).astype(np.int8)

def ppu_model(x, mult, shift, zp, bias):
    val = (x + bias) * mult
    val = val >> shift
    val = val + zp
    val = np.clip(val, -128, 127)
    return val.astype(np.int8)

def pack_input_tile(a_tile):
    bytes_list = []
    for r in range(a_tile.shape[0]):
        for c in range(ARRAY_ROW):
            bytes_list.append(to_u8(a_tile[r, c]))
    words = []
    for i in range(0, len(bytes_list), 8):
        w = 0
        for b_i in range(8):
            w |= (bytes_list[i + b_i] << (8 * b_i))
        words.append(np.uint64(w))
    return words

def pack_weight_tile(b_tile):
    words = []
    for r in range(ARRAY_ROW):
        val128 = 0
        for c in range(ARRAY_COL):
            val = to_u8(b_tile[r, c])
            val128 |= (val << (8 * c))
        low = val128 & 0xFFFFFFFFFFFFFFFF
        high = (val128 >> 64) & 0xFFFFFFFFFFFFFFFF
        words.append(np.uint64(low))
        words.append(np.uint64(high))
    return words

def unpack_output_tile(words, m_pad):
    out = np.zeros((m_pad, ARRAY_COL), dtype=np.int8)
    for r in range(m_pad):
        low = int(words[2*r])
        high = int(words[2*r + 1])
        val128 = low | (high << 64)
        bytes_row = [(val128 >> (8*c)) & 0xFF for c in range(ARRAY_COL)]
        out[r, :] = u8_to_i8(bytes_row)
    return out

def softmax_rowwise(x):
    # x: [M, N] float32
    x_max = np.max(x, axis=1, keepdims=True)
    e = np.exp(x - x_max)
    return e / np.sum(e, axis=1, keepdims=True)


In [ ]:
BITSTREAM = './deit/deit_accel.bit'
IP_NAME = 'deit_accelerator_top_0'
DMA_NAME = 'axi_dma_0'

REG_CTRL      = 0x00
REG_STATUS    = 0x04
REG_CFG_SEQ   = 0x08
REG_CFG_ACC   = 0x0C
REG_VERSION   = 0x10
REG_PPU_MULT  = 0x14
REG_PPU_SHIFT = 0x18
REG_PPU_ZP    = 0x1C
REG_PPU_BIAS  = 0x20
REG_OUT_EN    = 0x24

REG_DBG_SNAP  = 0x28
REG_DBG_CLR   = 0x2C
REG_DBG0      = 0x30
REG_DBG1      = 0x34
REG_DBG2      = 0x38
REG_DBG3      = 0x3C

overlay = pynq.Overlay(BITSTREAM)
axi_ctrl = getattr(overlay, IP_NAME)
dma = getattr(overlay, DMA_NAME)

def axi_write(addr, val):
    axi_ctrl.write(addr, int(val))

def axi_read(addr):
    return axi_ctrl.read(addr)

def soft_reset():
    axi_write(REG_CTRL, 0x00)
    time.sleep(0.01)
    axi_write(REG_CTRL, 0x02)
    time.sleep(0.01)

def start_pulse():
    axi_write(REG_CTRL, 0x03)
    axi_write(REG_CTRL, 0x02)

def dbg_snap():
    axi_write(REG_DBG_SNAP, 1)

def dbg_clr():
    axi_write(REG_DBG_CLR, 1)

def dbg_read():
    dbg_snap()
    d0 = axi_read(REG_DBG0)
    return {
        'DBG0': d0,
        'state': d0 & 0x7,
        'dma_req': (d0 >> 3) & 0x1,
    }

def dma_status(ch):
    return ch._mmio.read(0x04)

def wait_dma_idle(ch, timeout=2.0):
    t0 = time.time()
    while time.time() - t0 < timeout:
        sr = dma_status(ch)
        if sr & 0x2:
            return True, sr
        time.sleep(0.001)
    return False, dma_status(ch)

print('[INFO] Overlay loaded')
print('VERSION = 0x%08x' % axi_read(REG_VERSION))


In [ ]:
def run_gemm_hw(a_int8, b_int8, ppu_mult, ppu_shift, ppu_zp, ppu_bias, timeout=3.0):
    # a: [M,K], b: [K,N]
    m, k = a_int8.shape
    k2, n = b_int8.shape
    assert k2 == k
    m_pad, k_pad, n_pad = calc_pad(m, k, n)
    k_tiles = k_pad // ARRAY_ROW
    n_tiles = n_pad // ARRAY_COL

    a_pad = np.zeros((m_pad, k_pad), dtype=np.int8)
    b_pad = np.zeros((k_pad, n_pad), dtype=np.int8)
    a_pad[:m, :k] = a_int8
    b_pad[:k, :n] = b_int8

    soft_reset()
    dbg_clr()
    axi_write(REG_CFG_SEQ, m_pad)
    axi_write(REG_CFG_ACC, 0)
    axi_write(REG_PPU_MULT, ppu_mult)
    axi_write(REG_PPU_SHIFT, ppu_shift)
    axi_write(REG_PPU_ZP, ppu_zp)
    axi_write(REG_PPU_BIAS, ppu_bias)
    axi_write(REG_OUT_EN, 0)

    c_hw = np.zeros((m_pad, n_pad), dtype=np.int8)

    for n_idx in range(n_tiles):
        for k_idx in range(k_tiles):
            acc_mode = 0 if (k_idx == 0) else 1
            out_en = 1 if (k_idx == k_tiles - 1) else 0
            axi_write(REG_CFG_ACC, acc_mode)
            axi_write(REG_OUT_EN, out_en)

            a_tile = a_pad[:, k_idx*ARRAY_ROW:(k_idx+1)*ARRAY_ROW]
            in_words = pack_input_tile(a_tile)
            buf_A = pynq.allocate(shape=(len(in_words),), dtype=np.uint64)
            buf_A[:] = in_words
            buf_A.flush()
            dma.sendchannel.transfer(buf_A)
            ok_mm2s, _ = wait_dma_idle(dma.sendchannel, timeout=timeout)
            if not ok_mm2s:
                raise RuntimeError('MM2S not idle after A preload')

            if out_en:
                out_words = m_pad * 2
                buf_C = pynq.allocate(shape=(out_words,), dtype=np.uint64)
                dma.recvchannel.transfer(buf_C)
            else:
                buf_C = None

            start_pulse()

            t0 = time.time()
            while time.time() - t0 < timeout:
                if dbg_read()['dma_req'] == 1:
                    break
                time.sleep(0.0005)
            else:
                raise RuntimeError('DMA_REQ timeout')

            b_tile = b_pad[k_idx*ARRAY_ROW:(k_idx+1)*ARRAY_ROW, n_idx*ARRAY_COL:(n_idx+1)*ARRAY_COL]
            w_words = pack_weight_tile(b_tile)
            buf_B = pynq.allocate(shape=(len(w_words),), dtype=np.uint64)
            buf_B[:] = w_words
            buf_B.flush()
            dma.sendchannel.transfer(buf_B)
            ok_mm2s2, _ = wait_dma_idle(dma.sendchannel, timeout=timeout)
            if not ok_mm2s2:
                raise RuntimeError('MM2S not idle after weight')

            t0 = time.time()
            done = False
            while time.time() - t0 < timeout:
                if (axi_read(REG_STATUS) & 0x1) != 0:
                    done = True
                    axi_write(REG_STATUS, 0x1)
                    break
                time.sleep(0.0005)
            if not done:
                raise RuntimeError('AP_DONE timeout')

            if out_en:
                ok_s2mm, _ = wait_dma_idle(dma.recvchannel, timeout=timeout)
                if not ok_s2mm:
                    raise RuntimeError('S2MM timeout')
                buf_C.invalidate()
                out_tile = unpack_output_tile(np.array(buf_C), m_pad)
                c_hw[:, n_idx*ARRAY_COL:(n_idx+1)*ARRAY_COL] = out_tile
                buf_C.close()

            buf_A.close()
            buf_B.close()

    return c_hw[:m, :n]

def run_gemm_sw(a_int8, b_int8, ppu_mult, ppu_shift, ppu_zp, ppu_bias):
    acc = a_int8.astype(np.int32) @ b_int8.astype(np.int32)
    return ppu_model(acc, ppu_mult, ppu_shift, ppu_zp, ppu_bias)


In [ ]:
# Load exported data
DATA_DIR = '../python/attn_data'
q_int8 = np.load(DATA_DIR + '/q_int8.npy')
k_int8 = np.load(DATA_DIR + '/k_int8.npy')
v_int8 = np.load(DATA_DIR + '/v_int8.npy')
scale_q = float(np.load(DATA_DIR + '/scale_q.npy'))
scale_k = float(np.load(DATA_DIR + '/scale_k.npy'))
scale_v = float(np.load(DATA_DIR + '/scale_v.npy'))
meta = np.load(DATA_DIR + '/meta.npy', allow_pickle=True).item()

M = int(meta['m'])
Dh = int(meta['dh'])
print('M =', M, 'Dh =', Dh)
print('scale_q =', scale_q, 'scale_k =', scale_k, 'scale_v =', scale_v)

assert q_int8.shape == (M, Dh)
assert k_int8.shape == (M, Dh)
assert v_int8.shape == (M, Dh)


In [ ]:
# GEMM1: Score = Q * K^T
PPU1 = {'mult': 1, 'shift': 8, 'zp': 0, 'bias': 0}
print('Running GEMM1 (QK^T)...')
score_hw = run_gemm_hw(q_int8, k_int8.T, PPU1['mult'], PPU1['shift'], PPU1['zp'], PPU1['bias'], timeout=3.0)
score_sw = run_gemm_sw(q_int8, k_int8.T, PPU1['mult'], PPU1['shift'], PPU1['zp'], PPU1['bias'])
print('GEMM1 done')


In [ ]:
# Softmax on PS
# Dequantize score
score_float_hw = score_hw.astype(np.float32) * (scale_q * scale_k * (2 ** PPU1['shift']))
score_float_sw = score_sw.astype(np.float32) * (scale_q * scale_k * (2 ** PPU1['shift']))

score_float_hw = score_float_hw / np.sqrt(float(Dh))
score_float_sw = score_float_sw / np.sqrt(float(Dh))

softmax_hw = softmax_rowwise(score_float_hw)
softmax_sw = softmax_rowwise(score_float_sw)

# Quantize softmax to int8 [0,127]
s_q_hw = np.clip(np.rint(softmax_hw * 127.0), 0, 127).astype(np.int8)
s_q_sw = np.clip(np.rint(softmax_sw * 127.0), 0, 127).astype(np.int8)
print('Softmax done')


In [ ]:
# GEMM2: Output = Softmax * V
PPU2 = {'mult': 1, 'shift': 12, 'zp': 0, 'bias': 0}
print('Running GEMM2 (Softmax * V)...')
out_hw = run_gemm_hw(s_q_hw, v_int8, PPU2['mult'], PPU2['shift'], PPU2['zp'], PPU2['bias'], timeout=3.0)
out_sw = run_gemm_sw(s_q_sw, v_int8, PPU2['mult'], PPU2['shift'], PPU2['zp'], PPU2['bias'])
print('GEMM2 done')


In [ ]:
# Compare
diff = out_hw.astype(np.int16) - out_sw.astype(np.int16)
max_diff = int(np.max(np.abs(diff)))
mismatch = int(np.sum(out_hw != out_sw))
print('max_diff =', max_diff)
print('mismatch =', mismatch)
if mismatch == 0:
    print('[PASS] Attention operator verified')
else:
    print('[FAIL] Attention operator mismatch')
